# Baseline: K-Means và DBSCAN

Chạy K-Means và DBSCAN trên 4 bộ dữ liệu 2D (`blobs`, `moons`, `circles`, `anisotropic`, `n_samples=300`, `seed=42`) đã chuẩn hóa qua `StandardScaler`.

In [1]:
import numpy as np

from spectral_clustering.baselines import run_dbscan, run_kmeans
from spectral_clustering.datasets import make_synthetic
from spectral_clustering.preprocessing import standardize_features

In [2]:
DATASET_CONFIGS = [
    ("blobs", 3),
    ("moons", 2),
    ("circles", 2),
    ("anisotropic", 3),
]

datasets_raw = {}
datasets_scaled = {}

for name, n_clusters in DATASET_CONFIGS:
    X, y = make_synthetic(name, seed=42)
    X_check, y_check = make_synthetic(name, seed=42)
    assert np.array_equal(X, X_check) and np.array_equal(y, y_check)
    assert X.shape == (300, 2) and y.shape == (300,)

    X_scaled = standardize_features(X)
    datasets_raw[name] = (X, y, n_clusters)
    datasets_scaled[name] = X_scaled
    print(f"{name:12s} | X_scaled={X_scaled.shape}, y={y.shape}, n_classes={len(np.unique(y))}")

blobs        | X_scaled=(300, 2), y=(300,), n_classes=3
moons        | X_scaled=(300, 2), y=(300,), n_classes=2
circles      | X_scaled=(300, 2), y=(300,), n_classes=2
anisotropic  | X_scaled=(300, 2), y=(300,), n_classes=3


In [3]:
EPS_GRID = [0.15, 0.25, 0.40]
MIN_SAMPLES_GRID = [5, 10]

dbscan_sweep = {}
for name, _ in DATASET_CONFIGS:
    X_scaled = datasets_scaled[name]
    rows = []
    print(f"=== DBSCAN grid: {name} ===")
    for eps in EPS_GRID:
        for min_samples in MIN_SAMPLES_GRID:
            labels = run_dbscan(X_scaled, eps=eps, min_samples=min_samples)
            n_found = len(set(labels) - {-1})
            n_noise = int(np.sum(labels == -1))
            noise_ratio = n_noise / len(labels)
            rows.append({
                "eps": eps,
                "min_samples": min_samples,
                "n_clusters": n_found,
                "n_noise": n_noise,
                "noise_ratio": noise_ratio,
            })
            print(
                f"  eps={eps:.2f}, min_samples={min_samples:2d} -> "
                f"clusters={n_found:2d}, noise={n_noise:3d} ({noise_ratio:.1%})"
            )
    dbscan_sweep[name] = rows

=== DBSCAN grid: blobs ===
  eps=0.15, min_samples= 5 -> clusters= 3, noise=  7 (2.3%)
  eps=0.15, min_samples=10 -> clusters= 3, noise= 20 (6.7%)
  eps=0.25, min_samples= 5 -> clusters= 3, noise=  1 (0.3%)
  eps=0.25, min_samples=10 -> clusters= 3, noise=  3 (1.0%)
  eps=0.40, min_samples= 5 -> clusters= 3, noise=  0 (0.0%)
  eps=0.40, min_samples=10 -> clusters= 3, noise=  0 (0.0%)
=== DBSCAN grid: moons ===
  eps=0.15, min_samples= 5 -> clusters=14, noise= 19 (6.3%)
  eps=0.15, min_samples=10 -> clusters= 7, noise=210 (70.0%)
  eps=0.25, min_samples= 5 -> clusters= 2, noise=  0 (0.0%)
  eps=0.25, min_samples=10 -> clusters= 3, noise=  6 (2.0%)
  eps=0.40, min_samples= 5 -> clusters= 2, noise=  0 (0.0%)
  eps=0.40, min_samples=10 -> clusters= 2, noise=  0 (0.0%)
=== DBSCAN grid: circles ===
  eps=0.15, min_samples= 5 -> clusters=18, noise=123 (41.0%)
  eps=0.15, min_samples=10 -> clusters= 1, noise=290 (96.7%)
  eps=0.25, min_samples= 5 -> clusters= 7, noise=  0 (0.0%)
  eps=0.25, mi

In [4]:
SELECTED_DBSCAN = {
    "blobs": (0.25, 5),
    "moons": (0.25, 5),
    "circles": (0.40, 5),
    "anisotropic": (0.25, 5),
}

baseline_results = {}
for name, n_clusters in DATASET_CONFIGS:
    _, y, _ = datasets_raw[name]
    X_scaled = datasets_scaled[name]
    eps, min_samples = SELECTED_DBSCAN[name]

    kmeans_labels = run_kmeans(X_scaled, n_clusters=n_clusters, random_state=42)
    dbscan_labels = run_dbscan(X_scaled, eps=eps, min_samples=min_samples)

    assert kmeans_labels.shape == (300,) and dbscan_labels.shape == (300,)

    baseline_results[name] = {
        "X_scaled": X_scaled,
        "y": y,
        "kmeans_labels": kmeans_labels,
        "dbscan_labels": dbscan_labels,
        "n_clusters": n_clusters,
        "eps": eps,
        "min_samples": min_samples,
    }

    db_clusters = len(set(dbscan_labels) - {-1})
    db_noise = np.mean(dbscan_labels == -1)
    print(
        f"{name:12s} | K-Means(k={n_clusters}): {len(np.unique(kmeans_labels))} cụm | "
        f"DBSCAN(eps={eps}, min_samples={min_samples}): {db_clusters} cụm, noise={db_noise:.1%}"
    )

blobs        | K-Means(k=3): 3 cụm | DBSCAN(eps=0.25, min_samples=5): 3 cụm, noise=0.3%


moons        | K-Means(k=2): 2 cụm | DBSCAN(eps=0.25, min_samples=5): 2 cụm, noise=0.0%
circles      | K-Means(k=2): 2 cụm | DBSCAN(eps=0.4, min_samples=5): 2 cụm, noise=0.0%
anisotropic  | K-Means(k=3): 3 cụm | DBSCAN(eps=0.25, min_samples=5): 3 cụm, noise=0.0%


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from spectral_clustering.visualization import plot_baseline_comparison

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").is_file():
    raise RuntimeError("Hãy chạy notebook từ thư mục gốc repo hoặc notebooks/.")

output_dir = repo_root / "results" / "figures" / "baselines"

for name, result in baseline_results.items():
    eps = result["eps"]
    min_samples = result["min_samples"]
    title = f"{name} | DBSCAN eps={eps:g}, min_samples={min_samples}"

    fig, _ = plot_baseline_comparison(
        result["X_scaled"],
        result["kmeans_labels"],
        result["dbscan_labels"],
        y=result["y"],
        title=title,
        save_path=output_dir / f"{name}.png",
    )
    plt.show()
    plt.close(fig)